In [1]:
# === Config ===
MASTER_PKL = "/home/macula/SMATousi/Desktop/all_6q_new/trotter/all_circ_dict.pkl"   # the dict: {0: {'circ': ..., 'para': {'J':..., 'h':..., 't':...}, ...}, ...}
CIRCUIT_DIR = "/home/macula/SMATousi/Desktop/all_6q_new/trotter/all_noise/dataset/"  # folder that has 1200 per-circuit pkl files
OUT_PT = "/home/macula/SMATousi/Desktop/all_6q_new/trotter/all_6_trotter_graphs_1200.pt"          # output file

GLOB_PATTERNS = [
    "{i}.pkl",
    "data_{i}.pkl",
    "{i}/data_{i}.pkl",
    "{i}/graph.pkl",
    "circ_{i}.pkl",
]  # edit/extend if your filenames differ

import os, glob, pickle
from pathlib import Path
from typing import Any, Dict, Optional, List, Tuple, Union

import numpy as np
import torch
from torch_geometric.data import Data
from tqdm import tqdm

# --- Helpers ---

def load_pickle(p: Union[str, Path]) -> Any:
    with open(p, "rb") as f:
        return pickle.load(f)

def locate_per_circuit_pkl(i: int) -> Optional[Path]:
    base = Path(CIRCUIT_DIR)
    for pat in GLOB_PATTERNS:
        cand = base / pat.format(i=i)
        if cand.exists():
            return cand
    # last resort: try any file in a subfolder that contains the index in the name
    wild = list(base.glob(f"**/*{i}*.pkl"))
    return wild[0] if wild else None

def to_edge_index_from_adj(adj: np.ndarray) -> torch.Tensor:
    idx = np.argwhere(adj != 0)
    if idx.size == 0:
        return torch.zeros((2, 0), dtype=torch.long)
    return torch.from_numpy(idx.T).long()

def build_edge_attr_from_dict(edge_feature, edge_index: torch.Tensor) -> Optional[torch.Tensor]:
    if edge_feature is None:
        return None
    if not isinstance(edge_feature, dict):
        # Accept arrays too
        try:
            arr = np.asarray(edge_feature)
            if arr.ndim == 1:
                arr = arr[:, None]
            E = edge_index.shape[1]
            if arr.shape[0] != E:
                if arr.ndim == 2 and arr.shape[1] == E:
                    arr = arr.T
                else:
                    print(f"[WARN] edge_feature array shape {arr.shape} != num_edges {E}; dropping edge_attr.")
                    return None
            return torch.tensor(arr, dtype=torch.float32)
        except Exception:
            print("[WARN] edge_feature not dict/array; dropping edge_attr.")
            return None

    # infer feature dim F from first value
    F = None
    for v in edge_feature.values():
        if isinstance(v, (list, tuple, np.ndarray)):
            F = int(np.asarray(v, dtype=float).size)
        else:
            F = 1
        break
    if F is None:
        return None

    def as_int_pair(k):
        if isinstance(k, tuple) and len(k) == 2:
            return int(k[0]), int(k[1])
        if isinstance(k, str):
            try:
                u, v = eval(k)
                return int(u), int(v)
            except Exception:
                return None
        return None

    norm = {}
    for k, v in edge_feature.items():
        kk = as_int_pair(k)
        if kk is None:
            continue
        if isinstance(v, (list, tuple, np.ndarray)):
            vv = np.asarray(v, dtype=float).reshape(-1)
        else:
            vv = np.array([float(v)], dtype=float)
        if vv.size != F:
            vv = vv[:F] if vv.size > F else np.pad(vv, (0, F - vv.size))
        norm[kk] = vv

    E = edge_index.shape[1]
    out = np.zeros((E, F), dtype=float)
    src = edge_index[0].tolist()
    dst = edge_index[1].tolist()
    for e, (u, v) in enumerate(zip(src, dst)):
        out[e] = norm.get((u, v), 0.0)
    return torch.tensor(out, dtype=torch.float32)

def build_lightcone_masks(shortest_path, measured_node_idx, N: int, M: int) -> torch.Tensor:
    if shortest_path is None or measured_node_idx is None:
        return torch.ones((N, M), dtype=torch.bool)
    sp = np.asarray(shortest_path)
    meas = np.asarray(measured_node_idx).astype(int).reshape(-1)
    masks = np.zeros((N, M), dtype=bool)
    for m, node_id in enumerate(meas[:M]):
        if 0 <= node_id < N:
            col = sp[:, node_id]
            finite = np.isfinite(col) if np.issubdtype(col.dtype, np.floating) else (col >= 0)
            masks[:, m] = finite
    # If fewer indices than M, leave remaining columns as zeros (or set to ones if you prefer)
    return torch.from_numpy(masks)

def to_vec1(x) -> torch.Tensor:
    """Coerce y/noise_y to shape [1] float tensor."""
    if x is None:
        return None
    arr = np.asarray(x, dtype=float).reshape(-1)
    if arr.size == 0:
        return None
    return torch.tensor([float(arr[0])], dtype=torch.float32)




In [2]:
# --- Load master and convert all graphs ---

master = load_pickle(MASTER_PKL)  # dict: {0: {...}, 1: {...}, ...}
print("Master keys:", len(master))

graphs = []
missing = []

for i in tqdm(range(len(master))):
    meta = master[i]
    per_path = locate_per_circuit_pkl(i)
    if per_path is None:
        missing.append(i)
        continue

    rec = load_pickle(per_path)
    # Required-ish
    x = np.asarray(rec.get("x"), dtype=float)
    adj = np.asarray(rec.get("adj"))
    if x is None or adj is None:
        print(f"[WARN] #{i} missing x/adj; skip")
        continue
    N = x.shape[0]
    # optional positional enc.
    pe_feat = rec.get("pe_feat", None)
    if pe_feat is not None:
        pe = np.asarray(pe_feat, dtype=float)
        if pe.ndim == 1 and pe.shape[0] == N:
            pe = pe[:, None]
        if pe.ndim == 2 and pe.shape[0] == N:
            x = np.concatenate([x, pe], axis=1)

    x_t = torch.tensor(x, dtype=torch.float32)
    edge_index = to_edge_index_from_adj(adj)
    edge_attr_t = build_edge_attr_from_dict(rec.get("edge_feature", None), edge_index)

    # single measurement (M=1)
    y_t = to_vec1(rec.get("y", None))          # treat as ideal/ZNE target
    noisy_t = to_vec1(rec.get("noise_y", None))

    measured_idx = rec.get("node_idx", None)         # expects [1] or scalar
    shortest_path = rec.get("shortest_path", None)   # [N,N]
    lc_masks = build_lightcone_masks(shortest_path, measured_idx, N, M=1).bool()

    # meta from master + per-circuit
    para = meta.get("para", {}) if isinstance(meta, dict) else {}
    J = float(para.get("J")) if "J" in para else None
    h = float(para.get("h")) if "h" in para else None
    t = float(para.get("t")) if "t" in para else None
    step = rec.get("trotter_step", None)
    ctype = meta.get("circ_type", None)
    size = meta.get("size", None)

    d = Data(
        x=x_t,
        edge_index=edge_index,
        lightcone_masks=lc_masks,
    )
    if edge_attr_t is not None:
        d.edge_attr = edge_attr_t
    if y_t is not None:
        d.y = y_t                # [1]
    if noisy_t is not None:
        d.noisy_z = noisy_t      # [1]

    # attach metadata
    d.circ_idx = torch.tensor([i], dtype=torch.long)
    d.file_path = str(per_path)
    if step is not None:
        d.step = torch.tensor([float(step)], dtype=torch.float32)
    if J is not None: d.J = torch.tensor([J], dtype=torch.float32)
    if h is not None: d.h = torch.tensor([h], dtype=torch.float32)
    if t is not None: d.t = torch.tensor([t], dtype=torch.float32)
    if ctype is not None: d.circ_type = ctype
    if size is not None: d.size = torch.tensor([int(size)], dtype=torch.long)

    # quick sanity
    assert d.lightcone_masks.size(0) == d.x.size(0)
    assert d.lightcone_masks.size(1) == 1
    if hasattr(d, "edge_attr"):
        assert d.edge_attr.size(0) == d.edge_index.size(1)

    graphs.append(d)

print(f"Built {len(graphs)} graphs. Missing per-circuit files for {len(missing)} indices: {missing[:10]}{'...' if len(missing)>10 else ''}")

# Save once
torch.save(graphs, OUT_PT)
print("Saved:", Path(OUT_PT).resolve())

Master keys: 1200


100%|██████████████████████████████████████████████████████████████████████████████████████████████| 1200/1200 [00:24<00:00, 49.14it/s]


Built 1200 graphs. Missing per-circuit files for 0 indices: []
Saved: /home/macula/SMATousi/Desktop/all_6q_new/trotter/all_6_trotter_graphs_1200.pt
